**14/08/2026** -- Supplementary table S2: Proportion of variance explained by the five principal components.

Note that the figures being saved are for the 2000-2022 data.

In [1]:
library(dplyr)
library(readr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
root <- rprojroot::find_root(rprojroot::has_file(".gitignore"))
source(file.path(root, "src/deprivation_pm_bhm/data_prep.R"))
source(file.path(root, "src/deprivation_pm_bhm/pca.R"))

In [3]:
SCRATCH_DIR <- Sys.getenv("SCRATCH_DIR")

In [4]:
# Prepare data
DATA_PATH   <- file.path(SCRATCH_DIR, 
                         "data/spatial/fire_pm_dep_paper_data",
                         "proc_data/df_af_annual_2000_2023.csv")

df <- read_csv(DATA_PATH) |> 
    filter(year <= 2022) |> 
    group_by(lon, lat) |>
    mutate(grid_id = cur_group_id()) |> # create grid_id for projecting SE vars
    ungroup()

df <- project_indicators(
    df, 
    list(edu_mean_years     = 2017,
         imp_san_access_pct = 2017,
         stunting_pct_u5    = 2017
    )
)

# Do PCA
pca_res <- compute_pca(
    df,
    method          = "ppca",
    se_indicators   = c("edu_mean_years", "imp_san_access_pct", "log_GDP_pc",
                        "child_dep_pct", "stunting_pct_u5"),
    n_pcs           = 5,
    scale           = TRUE,
    centre          = TRUE,
    seed            = 42,
    positive_vars   = c("child_dep_pct", "stunting_pct_u5"),
    negative_vars   = c("edu_mean_years", "imp_san_access_pct", "log_GDP_pc")
)

Rows: 994412 Columns: 74
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (11): geometry, ISO_A3, country, region, continent, INCOME_GRP, ECONOMY,...
dbl (63): lon, lat, total_PM25, total_O3, fire_PM25, fire_O3, fire_PM25_hu, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] "Projecting edu_mean_years forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0227167 (tol = 0.002, component 1)”
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”


[1] "Projecting imp_san_access_pct forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.00349425 (tol = 0.002, component 1)”


[1] "Projecting stunting_pct_u5 forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0278442 (tol = 0.002, component 1)”
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”
Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(se_indicators)

  # Now:
  data %>% select(all_of(se_indicators))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”
Warning message in pcaMethods::ppca(X, nPcs = n_pcs, seed = seed):
“stopped after max iterations, but rel_ch was > threshold”


In [6]:
# Proportion of variance explained
# Cumulative R2 is available from pcaMethods::pca; derive per-PC R2 from it
r2_cum      <- as.numeric(pca_res$pca@R2cum)
r2          <- diff(c(0, r2_cum))
pc_names    <- paste0("PC", seq_along(r2))
var_expl    <- dplyr::bind_rows(                                          
    tibble::as_tibble_row(setNames(r2, pc_names)),
    tibble::as_tibble_row(setNames(r2_cum, pc_names))
) %>% 
    mutate(Metric = c("Proportion of Variance", 
                      "Cumulative Proportion")) %>% 
    relocate(Metric)

In [8]:
var_expl

Metric,PC1,PC2,PC3,PC4,PC5
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Proportion of Variance,0.6368059,0.1299742,0.1064734,0.06577651,0.06097
Cumulative Proportion,0.6368059,0.7667801,0.8732535,0.93903000,1.00000


In [9]:
readr::write_csv(
    var_expl,
    file.path(root, "paper_results/figures/supplementary_tab_S3.csv"),
)